# 🎾 Padel Analytics — Sponsor Player Classification

**Project:** Padel Analytics BI Platform (FIP)  
**Objective:** Classify players into sponsorship tiers to help brands target the right profiles  
**Model type:** Multi-class Classification  
**Target variable:** Sponsorship tier — `Premium` / `Standard` / `Basic`

---

### Tier Definition

| Tier | Criteria | Sponsorship offer |
|------|----------|-------------------|
| 🥇 Premium | Top 33% ranking + strong win rate | High-value deal |
| 🥈 Standard | Mid 33% ranking + moderate performance | Mid-range deal |
| 🥉 Basic | Bottom 33% ranking + limited results | Entry-level deal |

---

### Pipeline Overview

| Step | Description |
|------|-------------|
| 0 | Environment Setup |
| 1 | Imports & Configuration |
| 2 | Data Loading |
| 3 | Data Cleaning & Merging |
| 4 | Target Engineering (Tier Labeling) |
| 5 | Exploratory Data Analysis (EDA) |
| 6 | Feature Engineering |
| 7 | Preprocessing |
| 8 | Model Training & Evaluation |
| 9 | Feature Importance |
| 10 | Sponsor Recommendation |

---
## Step 0 — Environment Setup

In [ ]:
import subprocess, sys

required = ['xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']

for pkg in required:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
        print(f'✅  {pkg} already installed')
    except ImportError:
        print(f'📦  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'✅  {pkg} installed')

---
## Step 1 — Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection    import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.ensemble           import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics            import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})

# ── Random seed ───────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print('✅  All libraries imported successfully.')

---
## Step 2 — Data Loading

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
PATH_TOURNAMENTS = 'full_tournaments.csv'
PATH_PLAYERS     = 'men-Players.csv'

# ── Load raw data ─────────────────────────────────────────────────────────────
df_tournaments_raw = pd.read_csv(PATH_TOURNAMENTS)
df_players_raw     = pd.read_csv(PATH_PLAYERS)

print(f'Tournaments dataset : {df_tournaments_raw.shape[0]:>5} rows × {df_tournaments_raw.shape[1]} columns')
print(f'Players dataset     : {df_players_raw.shape[0]:>5} rows × {df_players_raw.shape[1]} columns')

print('\n── Tournaments (first 5 rows) ──')
display(df_tournaments_raw.head())

print('\n── Players (first 5 rows) ──')
display(df_players_raw.head())

---
## Step 3 — Data Cleaning & Merging

In [ ]:
# ── Work on copies ────────────────────────────────────────────────────────────
df_t = df_tournaments_raw.copy()
df_p = df_players_raw.copy()

# ─────────────────────────────────────────────────────────────────────────────
# 3.1  Remove leftover header rows
# ─────────────────────────────────────────────────────────────────────────────
mask_dirty = df_t['Round Reached'] == 'Round Reached'
print(f'Header rows removed : {mask_dirty.sum()}')
df_t = df_t[~mask_dirty].reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3.2  Fix data types
# ─────────────────────────────────────────────────────────────────────────────
df_t['Matches Played'] = pd.to_numeric(df_t['Matches Played'], errors='coerce')
df_t['Matches Won']    = pd.to_numeric(df_t['Matches Won'],    errors='coerce')
df_t['Points Earned']  = pd.to_numeric(df_t['Points Earned'],  errors='coerce')
df_t['prizemoney']     = pd.to_numeric(df_t['prizemoney'],      errors='coerce')

# ─────────────────────────────────────────────────────────────────────────────
# 3.3  Standardise text columns
# ─────────────────────────────────────────────────────────────────────────────
for col in ['Player Name', 'Round Reached', 'categorie']:
    df_t[col] = df_t[col].str.strip()

df_p['player']  = df_p['player'].str.strip()
df_p['country'] = df_p['country'].str.strip()

# ─────────────────────────────────────────────────────────────────────────────
# 3.4  Round → numeric score
# ─────────────────────────────────────────────────────────────────────────────
ROUND_SCORE = {
    'Winner'       : 7,
    'Final'        : 6,
    'Semifinals'   : 5,
    'Quarterfinals': 4,
    'Round of 16'  : 3,
    'Round of 32'  : 2,
    'Round of 64'  : 1
}
df_t['round_score'] = df_t['Round Reached'].map(ROUND_SCORE)

# ─────────────────────────────────────────────────────────────────────────────
# 3.5  Aggregate tournament data at player level
# ─────────────────────────────────────────────────────────────────────────────
player_stats = df_t.groupby('Player Name').agg(
    tournaments_played     = ('Tournament Name', 'count'),
    total_matches_played   = ('Matches Played',  'sum'),
    total_matches_won      = ('Matches Won',     'sum'),
    total_points_earned    = ('Points Earned',   'sum'),
    avg_points_per_tourney = ('Points Earned',   'mean'),
    avg_round_score        = ('round_score',     'mean'),
    best_round_score       = ('round_score',     'max'),
    titles_won             = ('Round Reached',   lambda x: (x == 'Winner').sum()),
    finals_reached         = ('Round Reached',   lambda x: x.isin(['Winner', 'Final']).sum()),
    semis_reached          = ('Round Reached',   lambda x: x.isin(['Winner', 'Final', 'Semifinals']).sum()),
).reset_index()

player_stats['match_win_rate'] = (
    player_stats['total_matches_won'] /
    player_stats['total_matches_played'].replace(0, np.nan)
).fillna(0)

player_stats['title_rate'] = (
    player_stats['titles_won'] / player_stats['tournaments_played']
)

player_stats['final_rate'] = (
    player_stats['finals_reached'] / player_stats['tournaments_played']
)

# ─────────────────────────────────────────────────────────────────────────────
# 3.6  Merge with ranking data
# ─────────────────────────────────────────────────────────────────────────────
df_p_renamed = df_p.rename(columns={
    'player'  : 'Player Name',
    'points'  : 'ranking_points',
    'position': 'ranking_position',
    'move'    : 'ranking_move'
})

df = player_stats.merge(
    df_p_renamed[['Player Name', 'ranking_points', 'ranking_position', 'ranking_move', 'country']],
    on='Player Name', how='left'
)

# Fill missing ranking info with median
for col in ['ranking_points', 'ranking_position', 'ranking_move']:
    df[col] = df[col].fillna(df[col].median())

print(f'\nClean merged dataset : {df.shape[0]} players × {df.shape[1]} columns')
display(df.head())

---
## Step 4 — Target Engineering (Tier Labeling)

We define the sponsorship tier using a **composite score** based on:
- World ranking position (lower = better)
- Ranking points
- Win rate
- Title rate

Tiers are assigned using **percentile-based thresholds** (top 33% / mid 33% / bottom 33%).

In [ ]:
# ── Build composite sponsor score ─────────────────────────────────────────────
# Normalise each component to [0, 1] then combine

def min_max(series):
    rng = series.max() - series.min()
    return (series - series.min()) / rng if rng > 0 else series * 0

# Ranking position: invert so that rank 1 → high score
df['rank_score']   = 1 - min_max(df['ranking_position'])
df['points_score'] = min_max(df['ranking_points'])
df['winrate_score']= min_max(df['match_win_rate'])
df['title_score']  = min_max(df['title_rate'])

# Weighted composite (ranking weighted more for sponsors)
df['sponsor_score'] = (
    0.40 * df['rank_score']    +
    0.25 * df['points_score']  +
    0.20 * df['winrate_score'] +
    0.15 * df['title_score']
)

# ── Assign tiers based on percentile cutoffs ──────────────────────────────────
p33 = df['sponsor_score'].quantile(0.33)
p66 = df['sponsor_score'].quantile(0.66)

def assign_tier(score):
    if score >= p66:
        return 'Premium'
    elif score >= p33:
        return 'Standard'
    else:
        return 'Basic'

df['tier'] = df['sponsor_score'].apply(assign_tier)

print(f'Tier thresholds:')
print(f'  Premium  : sponsor_score >= {p66:.4f}')
print(f'  Standard : {p33:.4f} <= sponsor_score < {p66:.4f}')
print(f'  Basic    : sponsor_score < {p33:.4f}')
print(f'\nTier distribution:')
print(df['tier'].value_counts())

# Drop helper columns
df.drop(columns=['rank_score', 'points_score', 'winrate_score', 'title_score'], inplace=True)

---
## Step 5 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 5.1  Tier distribution ────────────────────────────────────────────────────
TIER_COLORS = {'Premium': '#2ecc71', 'Standard': '#3498db', 'Basic': '#e74c3c'}
TIER_ORDER  = ['Premium', 'Standard', 'Basic']

tier_counts = df['tier'].value_counts().reindex(TIER_ORDER)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(tier_counts.index, tier_counts.values,
              color=[TIER_COLORS[t] for t in tier_counts.index], width=0.5)
ax.bar_label(bars, padding=4, fontsize=11)
ax.set_ylabel('Number of players')
ax.set_title('Player distribution by sponsorship tier')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  Key features by tier ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

features = [
    ('ranking_position',    'World ranking position (lower = better)'),
    ('ranking_points',      'Ranking points'),
    ('match_win_rate',      'Match win rate'),
    ('title_rate',          'Title rate (titles / tournaments played)'),
]

for ax, (feat, label) in zip(axes.flat, features):
    data = [df[df['tier'] == t][feat].dropna() for t in TIER_ORDER]
    bp = ax.boxplot(data, labels=TIER_ORDER, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, tier in zip(bp['boxes'], TIER_ORDER):
        patch.set_facecolor(TIER_COLORS[tier])
        patch.set_alpha(0.7)
    ax.set_title(label)

plt.suptitle('Feature distribution by sponsorship tier', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3  Top 10 Premium players ───────────────────────────────────────────────
top_premium = (
    df[df['tier'] == 'Premium']
    .sort_values('sponsor_score', ascending=False)
    .head(10)[['Player Name', 'ranking_position', 'ranking_points',
               'match_win_rate', 'title_rate', 'titles_won', 'sponsor_score']]
    .reset_index(drop=True)
)
top_premium.index += 1
top_premium.index.name = 'Rank'

print('🥇 Top 10 Premium players')
display(top_premium)

---
## Step 6 — Feature Engineering

In [ ]:
# ── Define features ───────────────────────────────────────────────────────────
FEATURES = [
    # Ranking
    'ranking_points',
    'ranking_position',
    'ranking_move',
    # Win rate & titles
    'match_win_rate',
    'title_rate',
    'final_rate',
    'titles_won',
    'finals_reached',
    'semis_reached',
    # Volume
    'tournaments_played',
    'total_matches_played',
    'avg_points_per_tourney',
    'total_points_earned',
]

TARGET = 'tier'

# ── Encode target ─────────────────────────────────────────────────────────────
TIER_ENCODE = {'Premium': 2, 'Standard': 1, 'Basic': 0}
df['target'] = df[TARGET].map(TIER_ENCODE)

# ── Drop rows with missing features ──────────────────────────────────────────
df_model = df.dropna(subset=FEATURES + ['target']).copy()

X = df_model[FEATURES]
y = df_model['target'].astype(int)

print(f'Feature matrix shape : {X.shape}')
print(f'Target distribution  :')
print(y.value_counts().rename({2: 'Premium', 1: 'Standard', 0: 'Basic'}))

---
## Step 7 — Preprocessing

In [ ]:
# ── Train / Test split (stratified) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# ── Scale features ────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Convert back to DataFrame to keep feature names
X_train_sc = pd.DataFrame(X_train_sc, columns=FEATURES)
X_test_sc  = pd.DataFrame(X_test_sc,  columns=FEATURES)

print(f'Training set : {X_train_sc.shape[0]} rows')
print(f'Test set     : {X_test_sc.shape[0]} rows')

---
## Step 8 — Model Training & Evaluation

In [ ]:
# ── Define candidate models ───────────────────────────────────────────────────
models = {
    'Random Forest'    : RandomForestClassifier(
                            n_estimators=200, max_depth=10,
                            class_weight='balanced', random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(
                            n_estimators=200, max_depth=5,
                            learning_rate=0.05, random_state=SEED),
    'XGBoost'          : XGBClassifier(
                            n_estimators=200, max_depth=6, learning_rate=0.05,
                            eval_metric='mlogloss', random_state=SEED, verbosity=0)
}

# ── Cross-validated comparison ────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f'{name:<22}  CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

# ── Plot CV results ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([results[m] for m in models], labels=list(models.keys()),
           patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Accuracy (5-fold CV)')
ax.set_title('Model comparison — cross-validated accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train best model on full training set ─────────────────────────────────────
best_model_name = max(results, key=lambda m: results[m].mean())
print(f'Best model selected : {best_model_name}')

best_model = models[best_model_name]
best_model.fit(X_train_sc, y_train)

# ── Evaluate on test set ──────────────────────────────────────────────────────
y_pred = best_model.predict(X_test_sc)

acc = accuracy_score(y_test, y_pred)
print(f'\nTest accuracy : {acc:.4f}\n')

print(classification_report(y_test, y_pred,
      target_names=['Basic', 'Standard', 'Premium'], zero_division=0))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Basic', 'Standard', 'Premium'],
    cmap='Blues',
    ax=ax
)
ax.set_title(f'Confusion matrix — {best_model_name} (test set)')
plt.tight_layout()
plt.show()

---
## Step 9 — Feature Importance

In [ ]:
# ── Extract and plot feature importances ──────────────────────────────────────
importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2ecc71' if v >= feat_imp.quantile(0.66)
          else '#3498db' if v >= feat_imp.quantile(0.33)
          else '#e74c3c' for v in feat_imp.values]
bars = ax.barh(feat_imp.index, feat_imp.values, color=colors)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xlabel('Feature importance')
ax.set_title(f'Feature importance — {best_model_name}')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features for sponsor classification:')
print(feat_imp.sort_values(ascending=False).head(5).to_string())

---
## Step 10 — Sponsor Recommendation

Two tools:
1. **Classify a single player** — get their tier + confidence
2. **Get all players by tier** — full ranked list for sponsors

In [ ]:
DECODE_TIER = {2: '🥇 Premium', 1: '🥈 Standard', 0: '🥉 Basic'}

def classify_player(player_name: str) -> None:
    """
    Classify a single player and display their sponsorship tier with confidence.

    Parameters
    ----------
    player_name : str  Exact player name as it appears in the dataset.
    """
    row = df_model[df_model['Player Name'] == player_name]

    if row.empty:
        print(f'❌  Player "{player_name}" not found in dataset.')
        return

    X_player = scaler.transform(row[FEATURES])
    pred     = best_model.predict(X_player)[0]
    proba    = best_model.predict_proba(X_player)[0]

    print(f'\n Player         : {player_name}')
    print(f' Predicted tier : {DECODE_TIER[pred]}')
    print(f'\n Confidence breakdown:')
    for label, prob in zip(['Basic', 'Standard', 'Premium'], proba):
        bar = '█' * int(prob * 30)
        print(f'   {label:<10} {bar:<30} {prob*100:.1f}%')
    print(f'\n Key stats:')
    print(f'   World ranking  : #{int(row["ranking_position"].values[0])}')
    print(f'   Ranking points : {int(row["ranking_points"].values[0])}')
    print(f'   Win rate       : {row["match_win_rate"].values[0]*100:.1f}%')
    print(f'   Titles         : {int(row["titles_won"].values[0])}')
    print(f'   Title rate     : {row["title_rate"].values[0]*100:.1f}%')

In [ ]:
def get_players_by_tier(tier: str, top_n: int = 10) -> pd.DataFrame:
    """
    Return the top-N players in a given sponsorship tier ranked by sponsor score.

    Parameters
    ----------
    tier  : str  One of 'Premium', 'Standard', 'Basic'.
    top_n : int  Number of players to return.
    """
    X_all  = scaler.transform(df_model[FEATURES])
    preds  = best_model.predict(X_all)
    probas = best_model.predict_proba(X_all)

    tier_code   = {'Premium': 2, 'Standard': 1, 'Basic': 0}[tier]
    tier_col    = list(best_model.classes_).index(tier_code)

    result = df_model[['Player Name', 'ranking_position', 'ranking_points',
                        'match_win_rate', 'title_rate', 'titles_won',
                        'sponsor_score']].copy()

    result['predicted_tier']   = [DECODE_TIER[p] for p in preds]
    result['tier_confidence_%'] = (probas[:, tier_col] * 100).round(1)

    filtered = (
        result[result['predicted_tier'] == DECODE_TIER[tier_code]]
        .sort_values('sponsor_score', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    filtered.index     += 1
    filtered.index.name = 'Rank'

    print(f'\n{DECODE_TIER[tier_code]} players — top {top_n}')
    print('─' * 70)
    return filtered

In [ ]:
# ── Example 1 : classify a specific player ────────────────────────────────────
classify_player('Arturo Coello')

In [ ]:
# ── Example 2 : get all Premium players ──────────────────────────────────────
display(get_players_by_tier('Premium', top_n=10))

In [ ]:
# ── Example 3 : get all Standard players ─────────────────────────────────────
display(get_players_by_tier('Standard', top_n=10))

In [ ]:
# ── Example 4 : visualise tier distribution with confidence ───────────────────
X_all  = scaler.transform(df_model[FEATURES])
preds  = best_model.predict(X_all)
probas = best_model.predict_proba(X_all).max(axis=1)

viz_df = df_model[['Player Name', 'ranking_position', 'sponsor_score']].copy()
viz_df['tier']       = [DECODE_TIER[p] for p in preds]
viz_df['confidence'] = (probas * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
for tier, color in [('🥇 Premium', '#2ecc71'), ('🥈 Standard', '#3498db'), ('🥉 Basic', '#e74c3c')]:
    sub = viz_df[viz_df['tier'] == tier]
    ax.scatter(sub['ranking_position'], sub['sponsor_score'],
               label=tier, color=color, alpha=0.7, edgecolors='white', s=60)

ax.axhline(p66, color='#2ecc71', linestyle='--', linewidth=1, alpha=0.5, label=f'Premium threshold ({p66:.2f})')
ax.axhline(p33, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.5, label=f'Basic threshold ({p33:.2f})')
ax.set_xlabel('World ranking position')
ax.set_ylabel('Sponsor score')
ax.set_title('Player classification — sponsor score vs world ranking')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary

| Item | Value |
|------|-------|
| Total players classified | 194 |
| Features used | 13 |
| Tier labeling method | Composite score (ranking 40% + points 25% + win rate 20% + title rate 15%) |
| Output | Per-player tier (Premium / Standard / Basic) + confidence score |

### How sponsors can use this
- **Premium** → high-visibility deal, jersey branding, ambassador contracts
- **Standard** → mid-range partnership, social media collaboration
- **Basic** → entry-level deal, local event sponsorship, rising talent investment